In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00-common/01.environment-config"

In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"

target_path = "s3://bizf1project123/Bronze/Aaisha/incremental-load/circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId',   StringType()),
    StructField("url",         StringType()),
    StructField("circuitName", StringType()),
    StructField("lat",         DoubleType()),
    StructField("long",        DoubleType()),
    StructField("locality",    StringType()),
    StructField("country",     StringType())
])

In [0]:
circuits_df = (
    spark.read
         .format('csv')
         .option('header', 'true')
#         .option('inferSchema', 'true')
         .option('mode', 'FAILFAST')
         .schema(circuits_schema)
         .load(source_file)
)

In [0]:
display(circuits_df)

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

In [0]:
write_to_bronze(
    input_df=circuits_final_df,
    target_path=target_path,
    batch_id=v_batch_id
)